# Eval 2 — Confidence Gating Ablation

**Target**: `ReactionHistory.min_confidence` in `online/online_session.py`  
**Robot needed**: No  
**Reruns model**: No (replay-based)

Compares `min_confidence=0.0` (no gating) vs `min_confidence=0.55` (production)
across 30 videos, holding `cooldown_s=0.5` constant.

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

sys.path.insert(0, os.path.abspath('..'))
from eval_utils import INTENT_COLORS, load_all_sessions

results = pd.read_csv('results.csv')
print(f"Loaded {len(results)} rows — {results['video'].nunique()} videos, "
      f"conditions: {results['condition'].unique().tolist()}")
results.head()

## Cell 2 — Summary table

In [ ]:
summary = (
    results
    .groupby('condition')[['transition_rate', 'flip_rate', 'entropy',
                            'suppression_rate', 'mean_confidence_at_transition']]
    .agg(['mean', 'std'])
    .round(4)
)
print("Mean ± std across videos:")
summary

## Cell 3 — Scatter: state_confidence vs proposed_intent, all windows

In [ ]:
sessions = load_all_sessions('../session_csvs')

if sessions.empty:
    print("No session CSVs found — generate them first.")
else:
    CONF_THRESHOLD = 0.55
    sessions['gating_pass'] = sessions['state_confidence'] >= CONF_THRESHOLD

    fig, ax = plt.subplots(figsize=(10, 5))

    # Suppressed windows (below threshold)
    below = sessions[~sessions['gating_pass']]
    above = sessions[sessions['gating_pass']]

    ax.scatter(below['proposed_intent'], below['state_confidence'],
               color='red', alpha=0.4, s=20, label='suppressed (conf < 0.55)', zorder=2)
    ax.scatter(above['proposed_intent'], above['state_confidence'],
               color='green', alpha=0.4, s=20, label='passed (conf ≥ 0.55)', zorder=2)

    ax.axhline(CONF_THRESHOLD, color='navy', linestyle='--', linewidth=1.5,
               label=f'threshold = {CONF_THRESHOLD}')
    ax.set_xlabel('Proposed intent')
    ax.set_ylabel('state_confidence')
    ax.set_title('State confidence vs proposed intent — all windows, all 30 videos')
    ax.legend(fontsize=9)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

    pct_suppressed = (~sessions['gating_pass']).mean() * 100
    print(f"Overall suppression rate: {pct_suppressed:.1f}% of all windows across all videos")

## Cell 4 — Histogram: state_confidence at did_change=True windows

In [ ]:
if not sessions.empty:
    changed = sessions[sessions['did_change'] == True]['state_confidence']
    not_changed = sessions[sessions['did_change'] == False]['state_confidence']

    fig, ax = plt.subplots(figsize=(8, 4))
    bins = np.linspace(0, 1, 25)
    ax.hist(not_changed, bins=bins, alpha=0.6, color='#90C8E8',
            label='did_change=False (maintained)', density=True)
    ax.hist(changed, bins=bins, alpha=0.6, color='#E8A090',
            label='did_change=True (transition)', density=True)
    ax.axvline(0.55, color='navy', linestyle='--', linewidth=1.5, label='threshold=0.55')
    ax.set_xlabel('state_confidence')
    ax.set_ylabel('Density')
    ax.set_title('Confidence distribution at transition vs maintenance windows')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

## Cell 5 — Bar chart: suppression_rate per video (with_gating condition only)

In [ ]:
gated = results[results['condition'] == 'with_gating'].sort_values(
    'suppression_rate', ascending=False
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(gated)), gated['suppression_rate'],
       color='#D85A30', edgecolor='k', width=0.7)
ax.set_xticks(range(len(gated)))
ax.set_xticklabels(gated['video'].tolist(), rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Suppression rate  (low_confidence / n_windows)')
ax.set_title('Confidence suppression rate per video  (min_confidence=0.55)')
ax.axhline(gated['suppression_rate'].mean(), color='navy', linestyle='--',
           label=f"mean = {gated['suppression_rate'].mean():.3f}")
ax.legend()
plt.tight_layout()
plt.show()

## Cell 6 — Interpretation

### Findings

**Suppression rate** (fraction of windows gated out due to low confidence)
varies across videos, reflecting variability in how well the pipeline can
estimate the VA state from the input.  Videos with higher suppression rates
tend to have more ambiguous or noisy frames.

**Confidence at transitions** (Cell 4 histogram): transitions in the original
session (did_change=True) tend to occur at higher confidence values, confirming
that the pipeline's own gating already preferentially admits high-confidence
transitions.  With `min_confidence=0.0`, more low-confidence windows would
flip the effective intent, increasing the transition and flip rates.

### Caveat

`state_confidence` is a pipeline-internal metric — a product of baseline
stability score and trend agreement confidence.  It reflects how
*self-consistent* the pipeline's own estimates are, **not** how accurately
the pipeline reads the subject's actual emotional state.  A high-confidence
incorrect prediction is still incorrect.

Validating `state_confidence` as a proxy for ground-truth accuracy requires
a user study with annotated emotional ground truth (IRB pending).